# YOLO HAM10000 — تدريب وتشغيل على Colab

هذا الدفتر يحمّل المشروع من GitHub، ينزّل بيانات HAM10000، ثم يوفّر قسمين منفصلين:

1. **قسم التدريب:** إعادة تدريب النماذج المتوازنة  
2. **قسم التشغيل عبر Gradio:** اختيار النموذج ورفع صورة وتشغيل الاختبار من واجهة تفاعلية

**المستودع:** https://github.com/AlrgabeAh2025/yolo-testing-project

> شغّل Colab بـ **GPU**: `Runtime → Change runtime type → T4 GPU`.

## 0) الإعداد: استنساخ المشروع وتثبيت المتطلبات

In [ ]:
# إعدادات عامة — عدّل إن لزم
REPO_URL = "https://github.com/AlrgabeAh2025/yolo-testing-project.git"
PROJECT_DIR = "/content/yolo-testing-project"
USE_DRIVE = False  # True لحفظ النتائج على Google Drive
DRIVE_PROJECT = "/content/drive/MyDrive/yolo-testing-project"

import os, sys, subprocess
from pathlib import Path

def sh(cmd: str):
    print(">>", cmd)
    subprocess.run(cmd, shell=True, check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = DRIVE_PROJECT

if not Path(PROJECT_DIR).exists():
    sh(f"git clone --depth 1 {REPO_URL} {PROJECT_DIR}")
else:
    print("Project exists:", PROJECT_DIR)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(Path(PROJECT_DIR) / "scripts"))
print("CWD:", os.getcwd())

In [ ]:
# تثبيت الاعتماديات + التحقق من GPU
%pip -q install -U ultralytics opencv-python-headless pillow pandas tqdm PyYAML matplotlib seaborn scikit-learn gradio

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("تحذير: لا يوجد GPU. التدريب سيكون بطيئًا جدًا.")

## 1) تحميل البيانات وتحضير مجموعة YOLO

المسار الموصى به:
- ميتاداتا + أقنعة من Dataverse
- الصور من ISIC Archive API
- ثم `prepare_yolo_dataset.py --balance-train --malignant-ratio 1.2`

> لقسم Gradio فقط يمكن تخطي التنزيل الكامل إن كانت أوزان `best.pt` موجودة؛ لكن التقييم على test يحتاج البيانات.

In [ ]:
from pathlib import Path

RAW = Path("data/raw")
META = RAW / "metadata" / "HAM10000_metadata.csv"
MASKS = RAW / "masks"
IMAGES = RAW / "images"
PROCESSED = Path("data/processed")

FORCE_REDOWNLOAD = False
FORCE_REPREPARE = False
SKIP_DATA_DOWNLOAD = False  # True إن أردت Gradio فقط على صور مرفوعة بدون تنزيل HAM

if not SKIP_DATA_DOWNLOAD:
    need_meta = FORCE_REDOWNLOAD or not META.is_file()
    need_masks = FORCE_REDOWNLOAD or not MASKS.is_dir() or not any(MASKS.glob("*"))
    need_images = FORCE_REDOWNLOAD or not IMAGES.is_dir() or len(list(IMAGES.glob("*.jpg"))) < 10000

    if need_meta or need_masks:
        sh("python scripts/download_ham10000.py --source dataverse --skip-images")
    else:
        print("Metadata/masks موجودة.")

    if need_images:
        sh("python scripts/download_images_isic.py --workers 4")
    else:
        print(f"Images موجودة: {len(list(IMAGES.glob('*.jpg')))}")

    need_prepare = (
        FORCE_REPREPARE
        or not (PROCESSED / "labels" / "train").is_dir()
        or not any((PROCESSED / "labels" / "train").glob("*.txt"))
    )
    if need_prepare:
        sh("python scripts/prepare_yolo_dataset.py --balance-train --malignant-ratio 1.2")
    else:
        print("data/processed جاهزة.")

    sh("python scripts/verify_dataset.py")
else:
    print("SKIP_DATA_DOWNLOAD=True")

---
# الجزء أ — التدريب (Training)

شغّل خلايا هذا القسم فقط إذا أردت **إعادة تدريب** النماذج.

الوصفة المعتمدة: `configs/train_balanced.yaml`  
أسماء التشغيل: `yolov5su_balanced` / `yolov8s_balanced` / `yolov3-tinyu_balanced`

In [ ]:
TRAIN_CONFIG = "configs/train_balanced.yaml"
EPOCHS = None
BATCH = 16
WORKERS = 2

TRAIN_MODELS = [
    ("yolov5su", "yolov5su_balanced"),
    ("yolov8s", "yolov8s_balanced"),
    ("yolov3-tinyu", "yolov3-tinyu_balanced"),
]

RESUME_IF_EXISTS = True
print("سيتم تدريب:", [n for _, n in TRAIN_MODELS])

In [ ]:
from pathlib import Path

def train_one(model_key: str, run_name: str):
    last = Path("runs/detect") / run_name / "weights" / "last.pt"
    cmd = [
        "python", "scripts/train_yolo.py",
        "--model", model_key,
        "--config", TRAIN_CONFIG,
        "--name", run_name,
        "--batch", str(BATCH),
        "--workers", str(WORKERS),
        "--data", "configs/data.yaml",
    ]
    if EPOCHS is not None:
        cmd += ["--epochs", str(EPOCHS)]
    if RESUME_IF_EXISTS and last.is_file():
        cmd.append("--resume")
        print(f"Resume {run_name} from {last}")
    sh(" ".join(cmd))
    best = Path("runs/detect") / run_name / "weights" / "best.pt"
    print("best:", best, "exists=", best.is_file())
    return best

for model_key, run_name in TRAIN_MODELS:
    print("\n" + "=" * 60)
    print("Training", run_name)
    print("=" * 60)
    train_one(model_key, run_name)

print("\nانتهى التدريب.")

In [ ]:
# تقييم التعميم بعد التدريب (اختياري)
sh(
    "python scripts/test_generalization.py "
    "--models yolov5su_balanced yolov8s_balanced yolov3-tinyu_balanced "
    "--image-source processed"
)

---
# الجزء ب — التشغيل عبر Gradio

1. شغّل خلية تجهيز النماذج  
2. شغّل خلية تشغيل الواجهة  
3. من الواجهة: اختر النموذج → ارفع صورة → اضبط الثقة → اضغط **Predict**

In [ ]:
from pathlib import Path
import numpy as np
import cv2
from ultralytics import YOLO

CLASS_NAMES = {0: "benign (حميد)", 1: "malignant (خبيث)"}

AVAILABLE_WEIGHTS = {
    "yolov5su_balanced": Path("runs/detect/yolov5su_balanced/weights/best.pt"),
    "yolov8s_balanced": Path("runs/detect/yolov8s_balanced/weights/best.pt"),
    "yolov3-tinyu_balanced": Path("runs/detect/yolov3-tinyu_balanced/weights/best.pt"),
}

# تحقق من الأوزان المتاحة فقط
MODEL_CHOICES = [k for k, p in AVAILABLE_WEIGHTS.items() if p.is_file()]
assert MODEL_CHOICES, "لا توجد أوزان best.pt. درّب النماذج أولًا أو تأكد من استنساخ المستودع كاملًا."
print("النماذج المتاحة:", MODEL_CHOICES)

_model_cache = {}

def get_model(name: str) -> YOLO:
    if name not in _model_cache:
        path = AVAILABLE_WEIGHTS[name]
        print("Loading", path)
        _model_cache[name] = YOLO(str(path))
    return _model_cache[name]

def predict_image(image, model_name: str, conf: float, iou: float):
    if image is None:
        return None, "ارفع صورة أولًا."
    if model_name not in AVAILABLE_WEIGHTS or not AVAILABLE_WEIGHTS[model_name].is_file():
        return None, f"الأوزان غير موجودة للنموذج: {model_name}"

    model = get_model(model_name)
    # Gradio يعطي RGB numpy
    if isinstance(image, np.ndarray):
        rgb = image
    else:
        rgb = np.array(image)

    results = model.predict(
        source=rgb,
        conf=float(conf),
        iou=float(iou),
        imgsz=640,
        verbose=False,
    )
    r0 = results[0]
    plotted_bgr = r0.plot()
    plotted_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)

    if r0.boxes is None or len(r0.boxes) == 0:
        text = (
            f"النموذج: {model_name}\n"
            f"النتيجة: لا يوجد كشف فوق عتبة الثقة ({conf:.2f})"
        )
        return plotted_rgb, text

    confs = r0.boxes.conf.cpu().numpy()
    clss = r0.boxes.cls.cpu().numpy().astype(int)
    idx = int(np.argmax(confs))
    pred_cls = int(clss[idx])
    pred_conf = float(confs[idx])
    label = CLASS_NAMES.get(pred_cls, str(pred_cls))

    lines = [
        f"النموذج: {model_name}",
        f"التصنيف: {label}",
        f"الثقة: {pred_conf:.3f}",
        f"عدد الصناديق: {len(r0.boxes)}",
        f"conf={conf:.2f} | iou={iou:.2f}",
    ]
    # عرض كل الصناديق باختصار
    for i, (c, s) in enumerate(zip(clss, confs)):
        lines.append(f"  - box[{i}]: {CLASS_NAMES.get(int(c), int(c))} ({float(s):.3f})")

    return plotted_rgb, "\n".join(lines)

print("جاهز لتشغيل Gradio.")

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=predict_image,
    inputs=[
        gr.Image(type="numpy", label="صورة الآفة الجلدية"),
        gr.Dropdown(
            choices=MODEL_CHOICES,
            value=MODEL_CHOICES[0],
            label="اختر النموذج",
        ),
        gr.Slider(0.05, 0.95, value=0.25, step=0.05, label="عتبة الثقة (conf)"),
        gr.Slider(0.1, 0.95, value=0.50, step=0.05, label="IoU"),
    ],
    outputs=[
        gr.Image(type="numpy", label="النتيجة مع الصندوق"),
        gr.Textbox(label="تفاصيل التصنيف", lines=8),
    ],
    title="YOLO Skin Lesion Tester",
    description=(
        "اختر نموذجًا متوازنًا، ارفع صورة، ثم اضغط Submit للاختبار. "
        "الأصناف: benign (حميد) / malignant (خبيث)."
    ),
    allow_flagging="never",
)

# share=True يعطي رابطًا عامًا مؤقتًا من Colab
demo.launch(share=True, debug=False)

### (اختياري) تقييم نموذج على مجموعة الاختبار من الكود

ليس عبر Gradio — لتقرير المقاييس على كل صور test.

In [ ]:
EVAL_MODEL = MODEL_CHOICES[0]  # أو "yolov5su_balanced"
sh(
    f"python scripts/test_generalization.py --models {EVAL_MODEL} "
    "--image-source processed"
)

---
## حفظ النتائج على Drive (اختياري)

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_OUT = "/content/drive/MyDrive/yolo_runs_backup"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
    sh(f"rsync -a runs/detect/ {DRIVE_OUT}/detect/")
    sh(f"rsync -a runs/generalization/ {DRIVE_OUT}/generalization/")
    print("Saved to", DRIVE_OUT)
else:
    print("SAVE_TO_DRIVE=False")

---
### ملخص الاستخدام

| الهدف | ماذا تشغّل |
|---|---|
| إعداد المشروع + البيانات | خلايا القسم 0 و 1 |
| إعادة تدريب النماذج | خلايا **الجزء أ** |
| اختبار تفاعلي للنماذج | خلايا **الجزء ب (Gradio)** |
| تقييم كامل على test | خلية التقييم الاختيارية |

أفضل نموذج حسب نتائج المشروع: `yolov5su_balanced`.